## Import Packages

In [18]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns',100)

## Loading data

In [2]:
train=pd.read_pickle('../data/train_engineered.pkl')


In [3]:
test=pd.read_pickle('../data/train_engineered.pkl')

In [4]:
train.shape

(590540, 453)

In [5]:
test.shape

(590540, 453)

## Baseline Features for Analysis 

In [10]:
baseline_features = [
    "TransactionAmt",
    "TransactionAmt_log",
    "ProductCD",
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
    "addr1",
    "addr2",
    "dist1",
    "dist2",
    "P_emaildomain_group",
    "R_emaildomain_group",
    "P_emaildomain_suffix",
    "R_emaildomain_suffix",
    "match_email",
    "browser_group",
    "os_group",
    "DeviceType",
    "C1",
    "C2",
    "C13",
    "D1",
    "D10",
    "D15",
    "TransactionAmt_to_mean_card1",
    "TransactionAmt_to_std_card1",
    "TransactionAmt_to_mean_card4",
    "TransactionAmt_to_std_card4"
]

## Dependent & Independent Variables

In [11]:
X = train[baseline_features].copy()
y = train['isFraud'].copy()

## Identify Categorical & Numerical Variables

In [15]:
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(exclude='object').columns.tolist()

print(f'Categorical Columns {categorical_cols}')
print(f'Numerical Columns {numerical_cols}')

Categorical Columns ['ProductCD', 'card4', 'card6', 'P_emaildomain_group', 'R_emaildomain_group', 'P_emaildomain_suffix', 'R_emaildomain_suffix', 'browser_group', 'os_group', 'DeviceType']
Numerical Columns ['TransactionAmt', 'TransactionAmt_log', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'dist2', 'match_email', 'C1', 'C2', 'C13', 'D1', 'D10', 'D15', 'TransactionAmt_to_mean_card1', 'TransactionAmt_to_std_card1', 'TransactionAmt_to_mean_card4', 'TransactionAmt_to_std_card4']


## Encoding Categorical Columns

In [17]:
for col in categorical_cols:
    le=LabelEncoder()
    X[col]=X[col].astype(str)
    X[col]=le.fit_transform(X[col])

## Handling Missing Values

In [19]:
impute=SimpleImputer(strategy='median')
X_imputed=pd.DataFrame(impute.fit_transform(X),columns=X.columns)

## Train Test Split

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X_imputed,y,test_size=0.2,random_state=42,stratify=y)

In [23]:
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')

X_train shape: (472432, 31)
X_test shape: (118108, 31)


In [28]:
print(f'Training target distribution {y_train.value_counts(normalize=True)*100}')
print(f'Testing target distribution {y_test.value_counts(normalize=True)*100}')

Training target distribution isFraud
0    96.501084
1     3.498916
Name: proportion, dtype: float64
Testing target distribution isFraud
0    96.50066
1     3.49934
Name: proportion, dtype: float64


A stratified train-test split was used because the target variable is highly imbalanced. Stratification ensures that the fraud and non-fraud proportions remain similar in both training and testing sets.

### Feature Scaling

In [29]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.fit_transform(X_test)

Feature scaling was applied for Logistic Regression because it is sensitive to differences in feature scale. Tree-based models such as Decision Tree and Random Forest do not require scaling, but the scaled data will be used for Logistic Regression.

## Evaluation Functions

In [30]:
model_results = []
def evaluate_model(model_name, model, X_train_data, X_test_data, y_train, y_test):
    y_pred = model.predict(X_train_data)

    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test_data)[:,1]
        roc_auc= roc_auc_score(y_test, y_pred)
    else:
        roc_auc=np.nan


    accuracy=accuracy(y_test, y_pred)
    precision=precision(y_test, y_pred)
    recall=recall(y_test, y_pred)
    f1=f1_score(y_test, y_pred)
    cr=classification_report(y_test, y_pred)

    print(f'Model: {model_name}')
    print('=' * 40)
    print(f'Accuracy: {accuracy}')
    print(f'Precision: {precision}')
    print(f'Recall: {recall}')
    print(f'F1 Score: {f1} ')
    print(f'\n Classification Report:\n {cr}')

    ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.show()

    model.results.append('Model':model_name, 'Accuracy':accuracy, 'Precision':precision, 'Recall':recall)
    
    